# 03 — Đánh giá Chất lượng Tái tạo 3D

Notebook này minh họa cách tính các **metrics đánh giá** chất lượng:
- **NME** (Normalized Mean Error) — đánh giá độ chính xác landmarks
- **3D Reconstruction Error** — MSE, MAE, RMSE trên vertices
- **PSNR** — chất lượng ảnh texture
- **Benchmark** — thời gian xử lý trung bình

## Nội dung
1. Tính NME cho face landmarks
2. Tính lỗi tái tạo 3D
3. Tính PSNR cho texture
4. Benchmark thời gian xử lý
5. Vẽ biểu đồ so sánh

## 1. Import

In [ ]:
import sys
from pathlib import Path

ROOT = Path('.').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt

from src.eval.metrics import compute_nme, compute_3d_reconstruction_error, compute_psnr
from src.eval.benchmark import benchmark_pipeline
from src.utils.io import ensure_dir, list_image_files, load_json
from src.utils.logger import get_logger

logger = get_logger('notebook_03')
print('✅ Import thành công!')

## 2. Tính NME (Normalized Mean Error)

**NME** đo độ lệch trung bình của các landmarks được dự đoán so với ground truth,
chuẩn hóa theo khoảng cách 2 mắt (inter-ocular distance).

$$NME = \frac{1}{N} \sum_{i=1}^{N} \frac{\|p_i - \hat{p}_i\|_2}{d_{\text{IOD}}}$$

In [ ]:
# Tạo dữ liệu mẫu giả để minh họa
np.random.seed(42)
N = 68  # Số landmarks

# Ground truth landmarks
gt_landmarks = np.random.rand(N, 2) * 200  # pixel coordinates

# Dự đoán với noise nhỏ
noise_level = 3.0  # pixels
pred_landmarks = gt_landmarks + np.random.randn(N, 2) * noise_level

# Inter-ocular distance (khoảng cách 2 mắt)
iod = 60.0  # pixels

nme = compute_nme(pred_landmarks, gt_landmarks, normalize_factor=iod)
print(f'NME = {nme:.4f} ({nme*100:.2f}%)')
print(f'(Mục tiêu: NME < 0.05 là tốt trên benchmark 300W)')

## 3. Tính 3D Reconstruction Error

In [ ]:
# Tạo dữ liệu 3D mẫu
N_VERTICES = 5023  # Số vertices của FLAME model

gt_vertices = np.random.randn(N_VERTICES, 3)  # Ground truth mesh
pred_vertices = gt_vertices + np.random.randn(N_VERTICES, 3) * 0.01  # Predicted mesh

metrics = compute_3d_reconstruction_error(pred_vertices, gt_vertices)

print('📊 3D Reconstruction Error:')
for k, v in metrics.items():
    print(f'   {k:12s} = {v:.6f}')

## 4. Tính PSNR cho Texture

In [ ]:
import cv2

# Tạo texture mẫu
H, W = 512, 512
gt_texture = np.random.randint(0, 256, (H, W, 3), dtype=np.uint8)

# Thêm noise (mô phỏng lỗi texture)
noise = np.random.randint(-20, 20, (H, W, 3))
pred_texture = np.clip(gt_texture.astype(int) + noise, 0, 255).astype(np.uint8)

psnr = compute_psnr(pred_texture, gt_texture)
print(f'PSNR = {psnr:.2f} dB')
print('(PSNR > 30 dB thường là chất lượng tốt)')

## 5. Biểu đồ so sánh NME theo noise level

In [ ]:
noise_levels = np.linspace(0, 10, 20)
nme_values = []

for noise in noise_levels:
    pred = gt_landmarks + np.random.randn(N, 2) * noise
    nme_val = compute_nme(pred, gt_landmarks, normalize_factor=iod)
    nme_values.append(nme_val)

plt.figure(figsize=(10, 5))
plt.plot(noise_levels, nme_values, 'b-o', linewidth=2, markersize=5)
plt.axhline(y=0.05, color='r', linestyle='--', label='Ngưỡng NME=0.05')
plt.xlabel('Noise level (pixels)', fontsize=12)
plt.ylabel('NME', fontsize=12)
plt.title('NME vs Noise Level - DECA_DHMT', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
ensure_dir(Path('reports/figures'))
plt.savefig('reports/figures/nme_vs_noise.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Đã lưu biểu đồ: reports/figures/nme_vs_noise.png')

## 6. Benchmark thời gian xử lý

In [ ]:
import time

# Đo thời gian xử lý của từng bước
stages = ['Face Detection', 'MediaPipe Landmarks', 'DECA Encoder', 'FLAME Decoder', 'Texture Render']
# Thời gian mô phỏng (giây) - thực tế phụ thuộc vào phần cứng
times_cpu = [0.05, 0.12, 0.85, 0.23, 0.18]
times_gpu = [0.02, 0.08, 0.15, 0.08, 0.05]

x = np.arange(len(stages))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, times_cpu, width, label='CPU', color='steelblue', alpha=0.8)
bars2 = ax.bar(x + width/2, times_gpu, width, label='GPU (CUDA)', color='darkorange', alpha=0.8)

ax.set_xlabel('Pipeline Stage', fontsize=12)
ax.set_ylabel('Thời gian (giây)', fontsize=12)
ax.set_title('Benchmark thời gian xử lý từng stage - DECA_DHMT', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(stages, rotation=15, ha='right')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

# Thêm giá trị lên cột
for bar in bars1:
    ax.annotate(f'{bar.get_height():.2f}s',
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)
for bar in bars2:
    ax.annotate(f'{bar.get_height():.2f}s',
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('reports/figures/benchmark_stages.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Tổng thời gian CPU: {sum(times_cpu):.2f}s')
print(f'Tổng thời gian GPU: {sum(times_gpu):.2f}s')
print(f'Speedup GPU/CPU: {sum(times_cpu)/sum(times_gpu):.1f}x')